# pims

## Example 1: open a multi-page TIFF

In [ ]:
# Example 1: pims.open picks a TIFF reader and exposes the pages as a sequence of frames
import pathlib, tempfile
import numpy as np
import tifffile
import pims

work = pathlib.Path(tempfile.mkdtemp())
stack = (np.arange(4 * 8 * 6, dtype=np.uint16).reshape(4, 8, 6))
tifffile.imwrite(work / "stack.tif", stack, photometric="minisblack")  # 4 pages, not 4 colour samples
frames = pims.open(str(work / "stack.tif"))
assert len(frames) == 4 and frames.frame_shape == (8, 6)
assert np.array_equal(frames[2], stack[2])
print("Example 1:", type(frames).__name__, len(frames), "frames of", frames.frame_shape)

## Example 2: a folder of PNG files as one sequence

In [ ]:
# Example 2: ImageSequence reads files matching a glob in name order
import pathlib, tempfile
import imageio.v3 as iio
import numpy as np
import pims

work = pathlib.Path(tempfile.mkdtemp())
for i in range(3):
    iio.imwrite(work / f"img_{i:02d}.png", np.full((5, 7), 40 * i, dtype=np.uint8))
seq = pims.ImageSequence(str(work / "img_*.png"))
assert len(seq) == 3 and seq.frame_shape == (5, 7)
assert [int(f.mean()) for f in seq] == [0, 40, 80]
print("Example 2:", type(seq).__name__, len(seq), "files, means", [int(f.mean()) for f in seq])

## Example 3: each frame is an ndarray that knows its position

In [ ]:
# Example 3: pims.Frame is an ndarray subclass carrying frame_no and metadata
import numpy as np
import pims

frame = pims.Frame(np.zeros((2, 3)), frame_no=7, metadata={"exposure_ms": 12})
assert isinstance(frame, np.ndarray) and frame.frame_no == 7 and frame.metadata["exposure_ms"] == 12
doubled = frame + 1
assert doubled.frame_no == 7 and doubled.sum() == 6
print("Example 3:", type(frame).__name__, frame.shape, "frame_no", frame.frame_no, frame.metadata)

## Example 4: slicing is lazy and keeps frame numbers

In [ ]:
# Example 4: frames[::2] reads nothing until used, and each frame keeps its original number
import pathlib, tempfile
import numpy as np
import tifffile
import pims

work = pathlib.Path(tempfile.mkdtemp())
tifffile.imwrite(work / "t.tif", np.stack([np.full((4, 4), i, dtype=np.uint8) for i in range(6)]), photometric="minisblack")
frames = pims.open(str(work / "t.tif"))
every_other = frames[::2]
assert len(every_other) == 3 and [int(f[0, 0]) for f in every_other] == [0, 2, 4]
assert [f.frame_no for f in every_other] == [0, 2, 4]
print("Example 4:", len(every_other), "frames:", [f.frame_no for f in every_other])

## Example 5: pipelines process frames on demand

In [ ]:
# Example 5: @pims.pipeline and the built-in as_grey run per frame, only when a frame is read
import numpy as np
import pims
from slicerator import Slicerator

rgb = Slicerator([np.full((3, 3, 3), v, dtype=np.float64) for v in (0.0, 0.5, 1.0)])
grey = pims.as_grey(rgb)
assert len(grey) == 3 and grey[1].shape == (3, 3) and np.allclose(grey[1], 0.5)

@pims.pipeline
def invert(frame):
    return 1.0 - frame

assert np.allclose(invert(grey)[2], 0.0)
print("Example 5:", [round(float(g.mean()), 2) for g in grey], "inverted:", [round(float(g.mean()), 2) for g in invert(grey)])